# Exploring AMD Radeon GPU Memory Hierarchy with HIP

## 1. Introduction to the AMD GPU Memory Hierarchy

To write highly optimized code for AMD Radeon GPUs (or any modern GPU), understanding the memory hierarchy is essential. A GPU is a massive throughput engine, but its arithmetic logic units (ALUs) can only perform calculations as fast as data is fed to them.

The memory hierarchy is designed as a pyramid: the closer the memory is to the compute cores, the faster it is (lower latency, higher bandwidth), but the smaller its capacity. In HIP (which maps directly to AMD's hardware), we interact with four primary levels of memory:

1. **CPU RAM (Host Memory):**
* **Location:** Off-GPU, physically located on the motherboard.
* **Connection:** Connected to the GPU via the PCIe bus.
* **Characteristics:** Largest capacity (System RAM), but incredibly slow to access from the GPU. Transferring data across the PCIe bus has the highest latency and lowest bandwidth.

2. **GPU Global Memory (Device Memory / VRAM):**
* **Location:** On the GPU board (e.g., GDDR6 or HBM).
* **Characteristics:** Large capacity (typically 8GB to 192GB). Accessible by all threads across all Compute Units (CUs). It has high bandwidth (hundreds of GB/s to TB/s) but high latency (hundreds of clock cycles).

3. **GPU Shared Memory (Local Data Share - LDS):**
* **Location:** On-chip memory within a Compute Unit (CU).
* **Characteristics:** Small capacity (usually 64KB per CU). It is shared only among threads within the same Workgroup (Block). Because it is on-chip, it has extremely low latency and massive bandwidth. It is user-managed cache.

4. **GPU Registers (Vector General-Purpose Registers - VGPRs):**
* **Location:** Directly attached to the ALUs inside the SIMD units.
* **Characteristics:** The fastest memory available. Dedicated entirely to a single thread (private memory). Accessing a register generally takes 0 extra clock cycles (latency is hidden by the pipeline). However, using too many registers limits the number of threads that can run concurrently (occupancy).

## 2. Bandwidth vs. Latency

When measuring memory performance, we look at two metrics:

* **Bandwidth (Throughput):** How much data can be moved per second (measured in GB/s). Imagine this as the width of a highway.
* **Latency:** The time it takes for a single data request to be fulfilled (measured in nanoseconds or clock cycles). Imagine this as the speed limit on that highway.

In the following HIP code, we will measure the **Bandwidth** of Host-to-Device transfers, Global Memory, and Shared Memory. We will also proxy **Register latency/speed** by measuring the compute throughput (GFLOPs) when data is held entirely in registers.

In [ ]:
%%writefile host_main.cpp
#include <hip/hip_runtime.h>
#include <iostream>
#include <vector>
#include <iomanip>
#include <chrono>

// Include external HIP kernels from src directory
#include "src/MemoryLevelSpeedTest.hip"

#define CHECK(cmd) { \
    hipError_t err = (cmd); \
    if (err != hipSuccess) { \
        std::cerr << "HIP Error: " << hipGetErrorString(err) << " at line " << __LINE__ << std::endl; \
        exit(EXIT_FAILURE); \
    } \
}

#define SIZE_T(x) ((size_t)(x))

// Kernel for latency measurement
__global__ void latency_kernel(float* data, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        volatile float x = data[idx];
        data[idx] = x + 1.0f;
    }
}

// Kernel for register latency measurement
__global__ void register_latency_kernel(float* output, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        volatile float x = 1.0f;
        for (int i = 0; i < 1000; i++) {
            x = x * 1.001f + 0.001f;
        }
        output[idx] = x;
    }
}

void run_benchmark(size_t size_BYTES, int N) {
    size_t size = size_BYTES;

    std::cout << "\n========== Data Size: " << (size / (1024.0 * 1024.0)) << " MB ==========" << std::endl;

    // Allocate CPU RAM (Host)
    float *h_A = new float[N];
    float *h_B = new float[N];
    float *h_C = new float[N];
    for (int i = 0; i < N; i++) { h_A[i] = 1.0f; h_B[i] = 2.0f; }

    // Allocate GPU RAM (Global Device)
    float *d_A, *d_B, *d_C;
    CHECK(hipMalloc(&d_A, size));
    CHECK(hipMalloc(&d_B, size));
    CHECK(hipMalloc(&d_C, size));

    hipEvent_t start, stop;
    CHECK(hipEventCreate(&start));
    CHECK(hipEventCreate(&stop));
    float milliseconds = 0;

    // Test 1: PCIe Bandwidth AND Latency (Host to Device)
    CHECK(hipEventRecord(start));
    CHECK(hipMemcpy(d_A, h_A, size, hipMemcpyHostToDevice));
    CHECK(hipEventRecord(stop));
    CHECK(hipEventSynchronize(stop));
    CHECK(hipEventElapsedTime(&milliseconds, start, stop));

    double pcie_bw = (size / (1024.0 * 1024.0 * 1024.0)) / (milliseconds / 1000.0);
    double PCIe_latency_sec = milliseconds / 1000.0;
    std::cout << "\n[PCIe] CPU RAM <-> GPU RAM:" << std::endl;
    std::cout << "   Bandwidth: " << std::fixed << std::setprecision(2) << pcie_bw << " GB/s" << std::endl;
    std::cout << "   Latency:   " << std::fixed << std::setprecision(3) << (milliseconds * 1000.0) << " microseconds" << std::endl;

    // Test 2: Global Memory Bandwidth AND Latency
    CHECK(hipMemcpy(d_B, h_B, size, hipMemcpyHostToDevice));
    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    CHECK(hipEventRecord(start));
    global_mem_kernel<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);
    CHECK(hipEventRecord(stop));
    CHECK(hipEventSynchronize(stop));
    CHECK(hipEventElapsedTime(&milliseconds, start, stop));

    double global_bw = (3.0 * size / (1024.0 * 1024.0 * 1024.0)) / (milliseconds / 1000.0);
    std::cout << "\n[GPU Global Memory] VRAM:" << std::endl;
    std::cout << "   Bandwidth: " << std::fixed << std::setprecision(2) << global_bw << " GB/s" << std::endl;
    std::cout << "   Kernel Latency: " << std::fixed << std::setprecision(3) << (milliseconds * 1000.0) << " microseconds" << std::endl;

    // Test 3: Shared Memory (LDS) Operations - measure time only
    CHECK(hipEventRecord(start));
    shared_mem_kernel<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_C, N);
    CHECK(hipEventRecord(stop));
    CHECK(hipEventSynchronize(stop));
    CHECK(hipEventElapsedTime(&milliseconds, start, stop));
    std::cout << "\n[GPU Shared Memory] LDS:" << std::endl;
    std::cout << "   Kernel Time: " << std::fixed << std::setprecision(3) << milliseconds << " ms" << std::endl;

    // Test 4: Register / ALU Speed - measure latency
    CHECK(hipEventRecord(start));
    register_kernel<<<blocksPerGrid, threadsPerBlock>>>(d_C, N);
    CHECK(hipEventRecord(stop));
    CHECK(hipEventSynchronize(stop));
    CHECK(hipEventElapsedTime(&milliseconds, start, stop));

    double gflops = (N * 10000.0 * 4.0) / (milliseconds / 1000.0) / 1e9;
    std::cout << "\n[GPU Register] L1/Register:" << std::endl;
    std::cout << "   Math Throughput: " << std::fixed << std::setprecision(2) << gflops << " GFLOP/s" << std::endl;
    std::cout << "   Kernel Latency:  " << std::fixed << std::setprecision(3) << (milliseconds * 1000.0) << " microseconds" << std::endl;

    // Test 5: CPU RAM Latency (memory access latency)
    auto cpu_start = std::chrono::high_resolution_clock::now();
    for (int i = 0; i < N; i++) {
        volatile float x = h_A[i] + h_B[i];
        h_C[i] = x;
    }
    auto cpu_end = std::chrono::high_resolution_clock::now();
    double cpu_time_ms = std::chrono::duration<double, std::milli>(cpu_end - cpu_start).count();
    double cpu_latency_per_element = (cpu_time_ms * 1000.0) / N;
    std::cout << "\n[CPU RAM] Host Memory:" << std::endl;
    std::cout << "   Total Access Time: " << std::fixed << std::setprecision(3) << cpu_time_ms << " ms" << std::endl;
    std::cout << "   Avg Latency/Elem:  " << std::fixed << std::setprecision(6) << cpu_latency_per_element << " microseconds" << std::endl;

    // Test 6: Fine-grained latency measurement with small transfers
    size_t small_size = sizeof(float); // Single float transfer for latency
    float *h_small = new float[1];
    float *d_small;
    CHECK(hipMalloc(&d_small, small_size));

    int iterations = 100;
    float total_latency = 0;
    for (int i = 0; i < iterations; i++) {
        h_small[0] = static_cast<float>(i);
        CHECK(hipEventRecord(start));
        CHECK(hipMemcpy(d_small, h_small, small_size, hipMemcpyHostToDevice));
        CHECK(hipEventRecord(stop));
        CHECK(hipEventSynchronize(stop));
        CHECK(hipEventElapsedTime(&milliseconds, start, stop));
        total_latency += milliseconds;
    }
    double avg_transfer_latency = (total_latency / iterations) * 1000.0; // microseconds
    std::cout << "\n[Fine-grained Latency]" << std::endl;
    std::cout << "   PCIe Single Transfer: " << std::fixed << std::setprecision(3) << avg_transfer_latency << " microseconds" << std::endl;

    // GPU kernel launch latency
    CHECK(hipEventRecord(start));
    latency_kernel<<<1, 1>>>(d_small, 1);
    CHECK(hipEventRecord(stop));
    CHECK(hipEventSynchronize(stop));
    CHECK(hipEventElapsedTime(&milliseconds, start, stop));
    std::cout << "   GPU Kernel Launch:    " << std::fixed << std::setprecision(3) << (milliseconds * 1000.0) << " microseconds" << std::endl;

    // Cleanup
    delete[] h_A; delete[] h_B; delete[] h_C;
    CHECK(hipFree(d_A)); CHECK(hipFree(d_B)); CHECK(hipFree(d_C));
    CHECK(hipEventDestroy(start)); CHECK(hipEventDestroy(stop));
    delete[] h_small;
    CHECK(hipFree(d_small));
}

int main() {
    std::cout << "=== Memory Hierarchy Latency & Bandwidth Benchmark ===" << std::endl;
    std::cout << "Testing different data sizes..." << std::endl;

    // Three test sizes: 64MB, 1GB, 10GB
    struct {
        size_t size_BYTES;
        size_t N;
        std::string name;
    } test_cases[] = {
        {SIZE_T(64) * 1024 * 1024, SIZE_T(64) * 1024 * 1024 / sizeof(float), "64 MB"},
        {SIZE_T(1024) * 1024 * 1024, SIZE_T(1024) * 1024 * 1024 / sizeof(float), "1 GB"},
        {SIZE_T(10) * 1024 * 1024 * 1024, SIZE_T(10) * 1024 * 1024 * 1024 / sizeof(float), "10 GB"}
    };

#define SIZE_T(x) ((size_t)(x))

    // Check GPU memory before running large tests
    size_t free_mem, total_mem;
    CHECK(hipMemGetInfo(&free_mem, &total_mem));
    std::cout << "\nGPU Memory: " << (free_mem / (1024.0 * 1024.0 * 1024.0)) << " GB free / "
              << (total_mem / (1024.0 * 1024.0 * 1024.0)) << " GB total" << std::endl;

    for (auto &test : test_cases) {
        // Check if we have enough memory
        size_t required = test.size_BYTES * 3; // 3 arrays
        if (required > free_mem * 0.9) {
            std::cout << "\n[SKIPPED] " << test.name << " - Not enough GPU memory" << std::endl;
            continue;
        }
        run_benchmark(test.size_BYTES, test.N);
    }

    return 0;
}


Writing host_main.cpp


## 3. Compile and Run

Use the following commands to compile and run the benchmark:



In [1]:
! hipcc host_main.cpp -o memory_benchmark -I./src

In [2]:
! ./memory_benchmark

=== Memory Hierarchy Latency & Bandwidth Benchmark ===
Testing different data sizes...

GPU Memory: 0.330078 GB free / 23.9844 GB total

========== Data Size: 64 MB ==========

[PCIe] CPU RAM <-> GPU RAM:
   Bandwidth: 3.78 GB/s
   Latency:   16543.924 microseconds

[GPU Global Memory] VRAM:
   Bandwidth: 128.57 GB/s
   Kernel Latency: 1458.344 microseconds

[GPU Shared Memory] LDS:
   Kernel Time: 0.416 ms

[GPU Register] L1/Register:
   Math Throughput: 29420.88 GFLOP/s
   Kernel Latency:  22809.944 microseconds

[CPU RAM] Host Memory:
   Total Access Time: 10.353 ms
   Avg Latency/Elem:  0.000617 microseconds

[Fine-grained Latency]
   PCIe Single Transfer: 28.913 microseconds
   GPU Kernel Launch:    294.122 microseconds

[SKIPPED] 1 GB - Not enough GPU memory

[SKIPPED] 10 GB - Not enough GPU memory


## 4. Analysis and Questions for Students

Once you run the code above, look at the console output and reflect on the following questions:

1. **The PCIe Bottleneck:** Look at the CPU to GPU bandwidth. It is likely between 10 to 30 GB/s depending on your PCIe generation (Gen3/Gen4/Gen5). How does this compare to the Global Memory bandwidth? *Takeaway: Avoid copying data between the CPU and GPU frequently. Keep data on the GPU as long as possible.*
2. **Global Memory Constraints:** You should see Global Memory bandwidth in the hundreds of GB/s. While fast, the memory bus can still easily become saturated if every mathematical operation requires a trip to VRAM.
3. **The Power of Shared Memory & Registers:** In Kernels 2 and 3, we execute thousands of iterations of math loops per thread. If those loops had to read from Global Memory, the kernel would take magnitudes longer. Because the variables are kept in Registers and LDS (which sit millimeters away from the ALUs inside the chip), the operations complete almost instantaneously.

Your Answer：

1. 

2. 

3.